
# Ensemble BO for f7–f8

Updated strategy:
- TuRBO-style global/local schedule
- ARD GP inside the trust region
- SVR permutation importance for trust-region scaling
- Local NN helper inside the trust region
- **Improved NN gradient proposals**: clipped + normalized + coordinate-weighted gradients, weak-dimension masking, noisy directional candidates
- Dimension-scaled trust-region distance


In [1]:

import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR, SVC, LinearSVC

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- Config ---
DATA_PATH = Path("data.csv")
FUNCTIONS = ["function_7", "function_8"]
DIMS = {"function_7": 6, "function_8": 8}
CURRENT_STEP = 9
N_GLOBAL = 8000
N_LOCAL = 3500
LOW, HIGH = 0.0, 1.0
RANDOM_SEED = 19
IS_MAXIMISATION = True
DEBUG = True
TR_INIT_RADIUS = 0.25
GLOBAL_STEP_FREQUENCY = 4
DISTANCE_BONUS = 0.04
NN_CANDIDATES = 18
DEVICE = "cpu"

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
try:
    torch.set_num_interop_threads(1)
    torch.set_num_threads(1)
except RuntimeError:
    pass
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass


def normal_pdf(z):
    return (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * z**2)


def normal_cdf(z):
    return 0.5 * (1.0 + np.vectorize(math.erf)(z / np.sqrt(2.0)))


def acquisition_ucb(mu, sigma, kappa):
    return mu + kappa * sigma


def acquisition_ei(mu, sigma, best_y, xi):
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - best_y - xi
    z = imp / sigma
    return imp * normal_cdf(z) + sigma * normal_pdf(z)


def read_data(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}.")
    df = pd.read_csv(path)
    df["function"] = df["function"].astype(str).str.strip()
    return df


def get_xy(df: pd.DataFrame, fname: str, d: int):
    sub = df[df["function"] == fname].copy()
    if sub.empty:
        raise ValueError(f"No rows found for function={fname} in {DATA_PATH}.")
    X = sub[[f"x{i}" for i in range(1, d + 1)]].to_numpy(dtype=float)
    y = sub["y"].to_numpy(dtype=float).reshape(-1)
    m = np.isfinite(X).all(axis=1) & np.isfinite(y)
    return X[m], y[m]


def format_output_line(fname, x):
    parts = [fname] + [f"{v:.6f}" for v in x.tolist()]
    return ", ".join(parts)


def minmax01(a):
    a = np.asarray(a, dtype=float)
    lo, hi = np.min(a), np.max(a)
    if hi - lo < 1e-12:
        return np.zeros_like(a)
    return (a - lo) / (hi - lo)


def safe_normalise(w, floor=1e-3):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.maximum(w, 0.0)
    if w.sum() <= 0:
        w = np.ones_like(w)
    w = w / w.sum()
    w = np.maximum(w, floor)
    return w / w.sum()


def weights_to_scales(w, clip=(0.35, 2.5)):
    w = safe_normalise(w)
    scales = np.sqrt(w * w.size)
    return np.clip(scales, clip[0], clip[1])


def scaled_distance(X, x0, w):
    X = np.asarray(X, dtype=float)
    x0 = np.asarray(x0, dtype=float)
    w = safe_normalise(w)
    diff = X - x0.reshape(1, -1)
    return np.sqrt(np.sum(w.reshape(1, -1) * diff * diff, axis=1))


def min_scaled_distance_to_observed(C, X, w):
    w = safe_normalise(w)
    diff = C[:, None, :] - X[None, :, :]
    d2 = np.sum(w.reshape(1, 1, -1) * diff * diff, axis=2)
    return np.sqrt(np.min(d2, axis=1))


def sample_uniform(n, d, rng):
    return rng.uniform(LOW, HIGH, size=(n, d))


def sample_local_weighted(best_x, n, rng, radius, dim_weights):
    scales = weights_to_scales(dim_weights)
    X = best_x.reshape(1, -1) + rng.normal(0.0, radius, size=(n, best_x.size)) * scales.reshape(1, -1)
    return np.clip(X, LOW, HIGH)


def pca_guided_candidates(X, n, rng, n_components=2, expand=0.75):
    d = X.shape[1]
    n_components = min(n_components, d)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    pca = PCA(n_components=n_components, random_state=0)
    Z = pca.fit_transform(Xs)
    z_lo = Z.min(axis=0)
    z_hi = Z.max(axis=0)
    span = np.where((z_hi - z_lo) == 0, 1.0, (z_hi - z_lo))
    Zc = rng.uniform(z_lo - expand * span, z_hi + expand * span, size=(n, n_components))
    Xc = scaler.inverse_transform(pca.inverse_transform(Zc))
    return np.clip(Xc, LOW, HIGH)


def extract_length_scales(kernel):
    if hasattr(kernel, "length_scale"):
        ls = np.asarray(kernel.length_scale, dtype=float)
        return ls.reshape(-1) if ls.ndim else np.array([float(ls)])
    for attr in ("k1", "k2", "kernel"):
        if hasattr(kernel, attr):
            ls = extract_length_scales(getattr(kernel, attr))
            if ls is not None:
                return ls
    return None


def ard_weights_from_gp(gp, d):
    ls = extract_length_scales(gp.kernel_)
    if ls is None:
        return np.ones(d) / d
    if ls.size == 1:
        ls = np.repeat(ls.item(), d)
    inv = 1.0 / np.maximum(ls[:d], 1e-8)
    return safe_normalise(inv)


def schedule_params(step: int):
    if 1 <= step <= 3:
        return "ucb", 6.0, None, "global"
    if (step % GLOBAL_STEP_FREQUENCY) == 0:
        return "ucb", 5.0, None, "global"
    return "ei", None, 0.02, "local"


def fit_gp(X, y, seed=0):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=(1e-5, 1e5),
        nu=2.5,
    ) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e-1))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=seed, n_restarts_optimizer=1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        gp.fit(Xs, y)
    return gp, scaler


def fit_local_svr(X_local, y_local):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_local)
    svr = SVR(kernel="rbf", C=10.0, epsilon=0.01, gamma="scale")
    svr.fit(Xs, y_local)
    return svr, scaler


def svr_importance_weights(svr, scaler, X_local, y_local):
    Xs = scaler.transform(X_local)
    result = permutation_importance(svr, Xs, y_local, n_repeats=20, random_state=RANDOM_SEED, scoring="r2")
    imp = np.maximum(result.importances_mean, 0.0)
    return safe_normalise(imp if imp.sum() > 0 else np.ones(X_local.shape[1]))


def trust_region_weights(gp, svr_w=None):
    ard_w = ard_weights_from_gp(gp, len(extract_length_scales(gp.kernel_)))
    if svr_w is None:
        return ard_w, ard_w
    combo = safe_normalise(0.6 * ard_w + 0.4 * svr_w)
    return combo, ard_w


class LocalMLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        hidden = 16 if d <= 6 else 24
        self.net = nn.Sequential(
            nn.Linear(d, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x)


def fit_local_nn(X_local, y_local, seed=RANDOM_SEED, epochs=250, lr=5e-3, weight_decay=1e-3):
    d = X_local.shape[1]
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    Xs = x_scaler.fit_transform(X_local).astype(np.float32)
    ys = y_scaler.fit_transform(y_local.reshape(-1, 1)).astype(np.float32)

    model = LocalMLP(d).to(DEVICE)
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    X_t = torch.tensor(Xs, dtype=torch.float32, device=DEVICE)
    y_t = torch.tensor(ys, dtype=torch.float32, device=DEVICE)

    model.train()
    last_loss = None
    for _ in range(epochs):
        opt.zero_grad(set_to_none=True)
        pred = model(X_t)
        loss = loss_fn(pred, y_t)
        loss.backward()
        opt.step()
        last_loss = float(loss.detach().cpu().item())

    return model, x_scaler, y_scaler, last_loss


def nn_gradient_at_point(model, x_scaler, y_scaler, x_raw):
    x_std = x_scaler.transform(np.asarray(x_raw, dtype=float).reshape(1, -1)).astype(np.float32)
    xt = torch.tensor(x_std, dtype=torch.float32, device=DEVICE, requires_grad=True)
    model.eval()
    out = model(xt)
    out.backward(torch.ones_like(out))
    grad_std = xt.grad.detach().cpu().numpy().reshape(-1)
    scale = np.asarray(x_scaler.scale_, dtype=float)
    grad_raw = grad_std / np.maximum(scale, 1e-12)
    return grad_raw


def safe_nn_direction(grad, dim_weights=None, eps=1e-12, clip_value=3.0):
    g = np.array(grad, dtype=float).copy()
    if dim_weights is not None:
        w = np.array(dim_weights, dtype=float)
        w = np.maximum(w, eps)
        w = w / (w.mean() + eps)
        g = g * w
    g = np.clip(g, -clip_value, clip_value)
    norm = np.linalg.norm(g)
    if norm < eps or (not np.isfinite(norm)):
        return np.zeros_like(g)
    return g / norm


def apply_dim_mask(direction, dim_weights, min_weight=0.15):
    w = np.array(dim_weights, dtype=float)
    w = w / (w.max() + 1e-12)
    mask = (w >= min_weight).astype(float)
    d = direction * mask
    norm = np.linalg.norm(d)
    if norm < 1e-12:
        return np.zeros_like(d)
    return d / norm


def nn_guided_candidates(
    x_best,
    grad,
    tr_radius,
    lb,
    ub,
    n_candidates=12,
    dim_weights=None,
    jitter_scale=0.15,
    step_fracs=(0.25, 0.5, 0.9),
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng(123)
    x_best = np.array(x_best, dtype=float)
    lb = np.array(lb, dtype=float)
    ub = np.array(ub, dtype=float)
    direction = safe_nn_direction(grad, dim_weights=dim_weights)
    if dim_weights is not None:
        direction = apply_dim_mask(direction, dim_weights, min_weight=0.15 if len(x_best) >= 8 else 0.10)

    candidates = []
    if np.allclose(direction, 0.0):
        for _ in range(n_candidates):
            jitter = rng.uniform(-1.0, 1.0, size=len(x_best))
            x = x_best + tr_radius * jitter
            x = np.clip(x, lb, ub)
            candidates.append(x)
        return np.array(candidates)

    scales = weights_to_scales(dim_weights if dim_weights is not None else np.ones_like(x_best))
    per_group = max(1, n_candidates // len(step_fracs))
    for alpha in step_fracs:
        for _ in range(per_group):
            jitter = rng.normal(0.0, jitter_scale, size=len(x_best)) * scales
            x = x_best + alpha * tr_radius * direction + jitter * tr_radius
            x = np.clip(x, lb, ub)
            candidates.append(x)

    while len(candidates) < n_candidates:
        jitter = rng.normal(0.0, jitter_scale, size=len(x_best)) * scales
        x = np.clip(x_best + 0.5 * tr_radius * direction + jitter * tr_radius, lb, ub)
        candidates.append(x)
    return np.array(candidates[:n_candidates])


def propose_next(fname, X, y, rng):
    mode, kappa, xi, phase = schedule_params(CURRENT_STEP)
    best_idx = int(np.argmax(y))
    best_x = X[best_idx].copy()
    best_y = float(y[best_idx])
    d = X.shape[1]

    gp, scaler = fit_gp(X, y, seed=RANDOM_SEED)
    ard_w = ard_weights_from_gp(gp, d)
    tr_weights = ard_w.copy()
    svr_w = None
    nn_loss = None
    use_nn_grad = False

    if phase == "global":
        C = sample_uniform(N_GLOBAL, d, rng)
    else:
        d_scaled = scaled_distance(X, best_x, ard_w)
        min_local = max(10, d + 2)
        mask = d_scaled <= TR_INIT_RADIUS
        X_loc = X[mask] if np.sum(mask) >= min_local else X
        y_loc = y[mask] if np.sum(mask) >= min_local else y

        try:
            svr, svr_scaler = fit_local_svr(X_loc, y_loc)
            svr_w = svr_importance_weights(svr, svr_scaler, X_loc, y_loc)
        except Exception:
            svr_w = None
        tr_weights, ard_w = trust_region_weights(gp, svr_w=svr_w)

        C_local = sample_local_weighted(best_x, N_LOCAL, rng, radius=TR_INIT_RADIUS, dim_weights=tr_weights)
        C_global = sample_uniform(N_GLOBAL // 5, d, rng)
        candidate_blocks = [C_local, C_global]

        # Local NN helper: only for directional proposals, coordinate reweighting,
        # and local refinement around incumbent best point.
        try:
            if len(X_loc) >= max(10, 2 * d):
                model, x_scaler_nn, y_scaler_nn, nn_loss = fit_local_nn(X_loc, y_loc)
                grad = nn_gradient_at_point(model, x_scaler_nn, y_scaler_nn, best_x)
                grad_norm = float(np.linalg.norm(grad))
                use_nn_grad = np.isfinite(grad_norm) and grad_norm > 1e-10 and np.isfinite(nn_loss)
                if use_nn_grad:
                    C_nn = nn_guided_candidates(
                        x_best=best_x,
                        grad=grad,
                        tr_radius=TR_INIT_RADIUS,
                        lb=np.full(d, LOW),
                        ub=np.full(d, HIGH),
                        n_candidates=NN_CANDIDATES,
                        dim_weights=tr_weights,
                        jitter_scale=0.15,
                        step_fracs=(0.2, 0.5, 0.85),
                        rng=rng,
                    )
                    candidate_blocks.append(C_nn)
        except Exception:
            use_nn_grad = False

        C = np.vstack(candidate_blocks)

    Xs = scaler.transform(C)
    mu, std = gp.predict(Xs, return_std=True)
    score = acquisition_ucb(mu, std, kappa=kappa) if mode == "ucb" else acquisition_ei(mu, std, best_y=best_y, xi=xi)
    score = score + DISTANCE_BONUS * minmax01(min_scaled_distance_to_observed(C, X, tr_weights))

    if DEBUG:
        extra = f" SVR={np.round(svr_w,3)}" if svr_w is not None else ""
        nn_msg = f" NN_grad=on loss={nn_loss:.4f}" if use_nn_grad and nn_loss is not None else " NN_grad=off"
        print(f"[{fname}] phase={phase} ARD={np.round(ard_w,3)} TR={np.round(tr_weights,3)}{extra}{nn_msg}")
    return C[int(np.argmax(score))]


df = read_data(DATA_PATH)
rng = np.random.default_rng(RANDOM_SEED)
for fname in FUNCTIONS:
    d = DIMS[fname]
    X, y = get_xy(df, fname, d)
    x_next = propose_next(fname, X, y, rng)
    print(format_output_line(fname, x_next))


[function_7] phase=local ARD=[0.001 0.001 0.001 0.308 0.441 0.248] TR=[0.051 0.041 0.054 0.236 0.408 0.21 ] SVR=[0.126 0.101 0.133 0.129 0.358 0.152] NN_grad=off
function_7, 0.251523, 0.443207, 0.787268, 0.974183, 0.022166, 0.063566
[function_8] phase=local ARD=[0.183 0.198 0.257 0.077 0.041 0.003 0.241 0.001] TR=[0.221 0.155 0.313 0.062 0.036 0.011 0.193 0.01 ] SVR=[0.278 0.091 0.396 0.04  0.027 0.022 0.122 0.023] NN_grad=on loss=0.0001
function_8, 0.076558, 0.000000, 0.000000, 0.183130, 1.000000, 0.700803, 0.305232, 0.897849
